## 평가

## 데이터 생성 셀

In [9]:
import json
import pandas as pd

print("====== 📊 AI HR 챗봇 [0, 1, 2 분기] 평가용 500개 데이터셋 생성 시작 ======\n")

eval_dataset = []

# ----------------------------------------------------------------
# [분류 0] 범위 초과 및 일상 질문 시나리오 -> 150개 생성
# ----------------------------------------------------------------
type_0_templates = [
    "오늘 {} 날씨 어때?", "{} 주변에 맛집 추천해줘", "{} 점심 메뉴 골라줘", 
    "{} 주가 지금 얼마야?", "{} 가는 비행기 표 예매해줘", "{}에서 제일 유명한 카페 알려줘"
]
keywords_0 = ["강남역", "홍대", "이태원", "판교", "성수동", "부산", "제주도", "여의도", "신촌", "가로수길", 
              "잠실", "건대", "명동", "종로", "대학로", "압구정", "해운대", "광안리", "대구", "대전", 
              "광주", "울산", "수원", "인천", "분당", "일산", "기흥", "동탄", "송도", "청라", "네이버", "카카오"]

count_0 = 0
while count_0 < 150:
    for kw in keywords_0:
        for template in type_0_templates:
            if count_0 >= 150: break
            eval_dataset.append({
                "query": template.format(kw),
                "expected_type": 0
            })
            count_0 += 1

# ----------------------------------------------------------------
# [분류 1] 자사 HR / 채용 데이터 질의 시나리오 -> 200개 생성
# ----------------------------------------------------------------
type_1_templates = [
    "우리 회사 {} 채용 공고 상태는?", "{} 직무 필수 역량이 뭐야?", 
    "{} 우대 기술 스택 가르쳐줘", "{} 채용 사유가 어떻게 돼?", 
    "{} 주요 업무 내용 요약해줄래?", "{} 근무 형태가 정규직이야?"
]
keywords_1 = ["프론트엔드 팀", "백엔드 개발자", "데이터 분석가", "마케터 신입", "신규 채용", "개발 직군"]

count_1 = 0
while count_1 < 200:
    for kw in keywords_1:
        for template in type_1_templates:
            if count_1 >= 200: break
            eval_dataset.append({
                "query": template.format(kw),
                "expected_type": 1
            })
            count_1 += 1

# ----------------------------------------------------------------
# [분류 2] 애플리케이션 사용법 / 기능 관련 시나리오 (RAG 매칭) -> 150개 생성
# ----------------------------------------------------------------
type_2_templates = [
    "{} 어디서 해?", "{} 버튼 어딨어?", "{} 메뉴 위치가 어디야?", 
    "{} 확인하려면 어디로 가야 돼?", "{} 방법 좀 가르쳐줘", "{} 하려는데 오류가 나요"
]
keywords_2 = ["자기소개서 업로드", "지원서 관리", "지원자 분석 리포트", "채용 공고 등록", "JD 관리", 
              "이전 채팅 기록", "대화 목록", "지원자 평가 결과", "평가 수정", "코멘트 변경"]

count_2 = 0
while count_2 < 150:
    for kw in keywords_2:
        for template in type_2_templates:
            if count_2 >= 150: break
            eval_dataset.append({
                "query": template.format(kw),
                "expected_type": 2
            })
            count_2 += 1

# 데이터 변환 및 저장
df = pd.DataFrame(eval_dataset)
df.to_json("chat_eval_dataset_v2.json", orient="records", force_ascii=False, indent=4)

print(f"✅ 데이터셋 구축 완료!")
print(f"👉 분류 0 (범위 밖): {len(df[df['expected_type']==0])}개")
print(f"👉 분류 1 (HR/채용): {len(df[df['expected_type']==1])}개")
print(f"👉 분류 2 (앱 사용법): {len(df[df['expected_type']==2])}개")
print(f"📁 총 {len(df)}개의 데이터 행이 'chat_eval_dataset_v2.json' 파일로 저장되었습니다.")

====== 📊 AI HR 챗봇 [0, 1, 2 분기] 평가용 500개 데이터셋 생성 시작 ======

✅ 데이터셋 구축 완료!
👉 분류 0 (범위 밖): 150개
👉 분류 1 (HR/채용): 200개
👉 분류 2 (앱 사용법): 150개
📁 총 500개의 데이터 행이 'chat_eval_dataset_v2.json' 파일로 저장되었습니다.


## 전수 벤치마크 평가 실행

In [13]:
import os
import sys
import json
import pandas as pd
from tqdm import tqdm
from copy import deepcopy

# 경로 세팅 (상황에 맞게 조절)
sys.path.append(os.path.abspath("."))
sys.path.append(os.path.abspath(".."))

from chat_agent import fall_case_node, context_extractor_node, hr_analyst_node, app_manual_rag_node

# ----------------------------------------------------------------
# 📊 [0, 1, 2 다중 분류 전용] 성능 평가 매트릭스 함수
# ----------------------------------------------------------------
def matrix_multiclass(correct_value: list[int], predict_value: list[int]):
    """
    3x3 Confusion Matrix 및 Accuracy, Precision, Recall, F1-Score(Macro) 출력
    """
    if len(correct_value) != len(predict_value):
        raise ValueError("두 리스트의 길이는 같아야 합니다.")

    # 3x3 매트릭스 초기화 (행: 실제값, 열: 예측값)
    # cm[actual][predicted]
    cm = {
        0: {0: 0, 1: 0, 2: 0},
        1: {0: 0, 1: 0, 2: 0},
        2: {0: 0, 1: 0, 2: 0}
    }

    # 매트릭스 카운트 채우기
    for correct, predict in zip(correct_value, predict_value):
        if correct in [0, 1, 2] and predict in [0, 1, 2]:
            cm[correct][predict] += 1
        else:
            # 에러 발생 등으로 예측값이 -1인 경우 등 예외 처리
            pass

    # 클래스별 지표 계산 (One-vs-Rest 방식)
    precisions = []
    recalls = []
    f1_scores = []
    total_correct = 0
    total_samples = len(correct_value)

    for i in [0, 1, 2]:
        tp = cm[i][i]
        total_correct += tp
        
        # Precision = TP / (TP + FP)
        fp = sum(cm[j][i] for j in [0, 1, 2] if j != i)
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        precisions.append(precision)
        
        # Recall = TP / (TP + FN)
        fn = sum(cm[i][j] for j in [0, 1, 2] if j != i)
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        recalls.append(recall)
        
        # F1 Score = 2 * P * R / (P + R)
        f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
        f1_scores.append(f1)

    # 전체 Accuracy (전체 맞춘 개수 / 전체 데이터 수)
    accuracy = total_correct / total_samples if total_samples > 0 else 0
    
    # Macro Average (각 클래스별 점수의 평균값)
    macro_precision = sum(precisions) / 3
    macro_recall = sum(recalls) / 3
    macro_f1 = sum(f1_scores) / 3

    # 3x3 Confusion Matrix 예쁘게 터미널 출력하기
    print("\n=============================================")
    print("📊 3x3 Confusion Matrix (다중 분류)")
    print("=============================================")
    print("                   Predicted")
    print("                 0       1       2")
    print(f"Actual 0 (범위)  {cm[0][0]:^5}   {cm[0][1]:^5}   {cm[0][2]:^5}")
    print(f"Actual 1 (채용)  {cm[1][0]:^5}   {cm[1][1]:^5}   {cm[1][2]:^5}")
    print(f"Actual 2 (앱법)  {cm[2][0]:^5}   {cm[2][1]:^5}   {cm[2][2]:^5}")
    print("=============================================")
    print(f"🎯 Accuracy  : {accuracy:.4f}")
    print(f"🎯 Precision : {macro_precision:.4f} (Macro Avg)")
    print(f"🎯 Recall    : {macro_recall:.4f} (Macro Avg)")
    print(f"🎯 F1 Score  : {macro_f1:.4f} (Macro Avg)")
    print("=============================================\n")


# ----------------------------------------------------------------
# ⚙️ 랑그래프 파이프라인 시뮬레이터 (동일 흐름 실행)
# ----------------------------------------------------------------
def run_chatbot_pipeline(user_query: str):
    state = {
        "chats": [user_query],
        "state": "START",
        "is_fall_case": None,
        "fall_case_type": None,
        "rag_search_query": "",
        "memories": [],
        "retrieved_manual_docs": [],
        "response": ""
    }
    state = fall_case_node(state)
    current_type = state.get("fall_case_type")
    
    if current_type == 1:
        state = context_extractor_node(state)
        state = hr_analyst_node(state)
    elif current_type == 2:
        state = app_manual_rag_node(state)
        
    return state


# ----------------------------------------------------------------
# 🚀 500개 벤치마크 테스트 평가 시작
# ----------------------------------------------------------------
if __name__ == "__main__":
    print("====== 🚀 500개 문항 4대 지표(평가셋 v2) 전수 검증 시작 ======")
    df_test = pd.read_json("chat_eval_dataset_v2.json")

    actual_list = []
    predict_list = []

    for idx, row in tqdm(df_test.iterrows(), total=len(df_test)):
        query = row["query"]
        expected = row["expected_type"]
        
        try:
            final_state = run_chatbot_pipeline(query)
            predicted = final_state.get("fall_case_type")
            
            # 예측값이 잘 담기지 않았다면 예외 처리용 기본값 세팅
            if predicted is None:
                predicted = -1
                
            actual_list.append(int(expected))
            predict_list.append(int(predicted))
            
        except Exception as e:
            actual_list.append(int(expected))
            predict_list.append(-1)  # 에러 발생 시 오답 처리

    # 최종 결과 매트릭스 함수 호출!
    matrix_multiclass(actual_list, predict_list)

====== 🚀 500개 문항 4대 지표(평가셋 v2) 전수 검증 시작 ======


  0%|          | 0/500 [00:00<?, ?it/s]

100%|██████████| 500/500 [25:48<00:00,  3.10s/it]  


📊 3x3 Confusion Matrix (다중 분류)
                   Predicted
                 0       1       2
Actual 0 (범위)   150      0       0  
Actual 1 (채용)    0      200      0  
Actual 2 (앱법)    0       2      148 
🎯 Accuracy  : 0.9960
🎯 Precision : 0.9967 (Macro Avg)
🎯 Recall    : 0.9956 (Macro Avg)
🎯 F1 Score  : 0.9961 (Macro Avg)

